# AI-Based Network Pathfinding & Attack Simulation System
**Course:** Artificial Intelligence (CSC 262)  
**Instructor:** Zeenat Zulfiqar  
**Institution:** COMSATS University Islamabad, Abbottabad Campus  

---
This notebook implements a network-based cyber attack simulation. A computer network is modelled as a weighted directed graph. An attacker starts at a designated entry node and attempts to reach a target system using classical and advanced AI search algorithms.

## 1. Install Dependencies

In [ ]:
# Install required visualization library
!pip install networkx matplotlib tabulate --quiet

## 2. Imports

In [ ]:
import heapq
import time
import random
import json
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import deque
from dataclasses import dataclass, field
from typing import Any
from tabulate import tabulate

print("All libraries loaded successfully.")

## 3. Network Graph Model

Nodes represent network entities (Workstations, Servers, Firewalls, Databases).  
Edges represent communication channels with weights encoding risk/vulnerability.  
Heuristics h(n) estimate remaining compromise steps to the Target Database.

In [ ]:
@dataclass
class SearchResult:
    """Unified result returned by all search algorithms."""
    algorithm: str
    path: list
    cost: float
    nodes_expanded: int
    execution_time_ms: float
    success: bool
    message: str = ""
    visited_order: list = field(default_factory=list)
    extra: dict = field(default_factory=dict)


class NetworkGraph:
    """
    Directed weighted graph for network topology simulation.
    adjacency : node -> list of (neighbor, weight)
    heuristics: node -> h(n) for informed search
    """

    def __init__(self):
        self.nodes = []
        self.adjacency = {}
        self.positions = {}
        self.heuristics = {}
        self.node_types = {}

    def add_node(self, name, x=0, y=0, h=0, ntype="host"):
        if name not in self.nodes:
            self.nodes.append(name)
            self.adjacency[name] = []
        self.positions[name] = (x, y)
        self.heuristics[name] = h
        self.node_types[name] = ntype

    def add_edge(self, u, v, weight=1.0):
        if u not in self.adjacency:
            self.add_node(u)
        if v not in self.adjacency:
            self.add_node(v)
        self.adjacency[u].append((v, weight))

    def neighbors(self, node):
        return self.adjacency.get(node, [])

    def h(self, node):
        return self.heuristics.get(node, 0.0)

    def reconstruct_path(self, came_from, goal):
        current = goal
        path = [current]
        while current in came_from:
            current = came_from[current]
            path.append(current)
        path.reverse()
        return path

    def path_cost(self, path):
        if len(path) < 2:
            return 0.0
        total = 0.0
        for i in range(len(path) - 1):
            u, v = path[i], path[i+1]
            for nbr, w in self.neighbors(u):
                if nbr == v:
                    total += w
                    break
            else:
                return float("inf")
        return total

print("NetworkGraph class defined.")

## 4. Network Topology Construction

The network models a realistic banking infrastructure with **10 nodes**:
- **Attacker Entry** – external threat actor origin (Start Node)
- Employee Workstation, ATM Server – initial access vectors
- Firewall – perimeter defence
- Authentication Server, Security Monitor – identity / visibility layer
- Main Transaction Server, Banking Database, Backup Server – core banking tier
- **Target Database** – crown-jewel asset (Goal Node)

Edge weights represent combined risk, detection probability, and traversal difficulty.

In [ ]:
def build_attack_network():
    """
    Banking network attack surface — 10 nodes, directed weighted edges.
    Heuristics h(n) = estimated number of compromise steps remaining to Target Database.
    """
    g = NetworkGraph()

    # name: (x, y, heuristic_h, node_type)
    layout = {
        "Attacker Entry":           (0.0, 0.5, 9, "attacker"),
        "Employee Workstation":     (0.2, 0.8, 8, "workstation"),
        "ATM Server":               (0.2, 0.2, 7, "atm"),
        "Firewall":                 (0.4, 0.5, 6, "firewall"),
        "Authentication Server":    (0.6, 0.8, 5, "auth"),
        "Security Monitor":         (0.6, 0.2, 5, "monitor"),
        "Main Transaction Server":  (0.75, 0.5, 3, "transaction"),
        "Banking Database":         (0.88, 0.8, 2, "database"),
        "Backup Server":            (0.88, 0.2, 4, "backup"),
        "Target Database":          (1.0, 0.5, 0, "target"),
    }

    for name, (x, y, h, ntype) in layout.items():
        g.add_node(name, x=x, y=y, h=h, ntype=ntype)

    edges = [
        # Initial access
        ("Attacker Entry",          "Employee Workstation",    2),
        ("Attacker Entry",          "ATM Server",              5),
        # Branch / DMZ to perimeter
        ("Employee Workstation",    "Firewall",                3),
        ("Employee Workstation",    "Security Monitor",        2),
        ("ATM Server",              "Firewall",                2),
        ("Security Monitor",        "Firewall",                1),
        # Perimeter to core
        ("Firewall",                "Authentication Server",   4),
        ("Firewall",                "Main Transaction Server", 7),
        ("Security Monitor",        "Main Transaction Server", 5),
        # Identity and transaction layer
        ("Authentication Server",   "Main Transaction Server", 2),
        ("Authentication Server",   "Banking Database",        6),
        ("Main Transaction Server", "Banking Database",        3),
        ("Main Transaction Server", "Target Database",         9),
        # Backup / alternate paths
        ("Main Transaction Server", "Backup Server",           2),
        ("Backup Server",           "Main Transaction Server", 3),
        ("Backup Server",           "Banking Database",        4),
        ("Banking Database",        "Target Database",         2),
    ]
    for u, v, w in edges:
        g.add_edge(u, v, w)

    return g


GRAPH = build_attack_network()
START = "Attacker Entry"
GOAL  = "Target Database"

print(f"Network built: {len(GRAPH.nodes)} nodes, {sum(len(v) for v in GRAPH.adjacency.values())} edges")
print(f"Start: {START}")
print(f"Goal : {GOAL}")

## 5. Network Visualization

In [ ]:
def visualize_network(graph, path=None, title="Network Topology", visited=None):
    """Draw the network graph. Highlights path in red if provided."""
    G = nx.DiGraph()
    for node in graph.nodes:
        G.add_node(node)
    for u in graph.nodes:
        for v, w in graph.neighbors(u):
            G.add_edge(u, v, weight=w)

    pos = {n: graph.positions[n] for n in graph.nodes}

    type_colors = {
        "attacker":    "#E24B4A",
        "target":      "#1D9E75",
        "firewall":    "#EF9F27",
        "workstation": "#378ADD",
        "atm":         "#7F77DD",
        "auth":        "#378ADD",
        "monitor":     "#EF9F27",
        "transaction": "#378ADD",
        "database":    "#7F77DD",
        "backup":      "#888780",
        "host":        "#888780",
    }
    node_colors = [type_colors.get(graph.node_types.get(n, "host"), "#888780") for n in G.nodes()]

    plt.figure(figsize=(14, 7))
    plt.title(title, fontsize=13, fontweight="bold", pad=12)

    # Draw all edges
    nx.draw_networkx_edges(G, pos, edge_color="#cccccc", arrows=True,
                           arrowsize=15, width=1.2, connectionstyle="arc3,rad=0.1")

    # Highlight path edges
    if path and len(path) > 1:
        path_edges = [(path[i], path[i+1]) for i in range(len(path)-1)]
        nx.draw_networkx_edges(G, pos, edgelist=path_edges, edge_color="#E24B4A",
                               arrows=True, arrowsize=20, width=3,
                               connectionstyle="arc3,rad=0.1")

    nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=1400, alpha=0.95)
    nx.draw_networkx_labels(G, pos, font_size=7, font_weight="bold", font_color="white")

    edge_labels = {(u, v): d["weight"] for u, v, d in G.edges(data=True)}
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=7, label_pos=0.35)

    legend_elements = [
        mpatches.Patch(color="#E24B4A", label="Attacker Entry"),
        mpatches.Patch(color="#1D9E75", label="Target Database"),
        mpatches.Patch(color="#EF9F27", label="Firewall / Monitor"),
        mpatches.Patch(color="#7F77DD", label="Database / ATM"),
        mpatches.Patch(color="#378ADD", label="Server / Auth"),
        mpatches.Patch(color="#888780", label="Backup / Host"),
    ]
    plt.legend(handles=legend_elements, loc="lower left", fontsize=8)
    plt.axis("off")
    plt.tight_layout()
    plt.show()


visualize_network(GRAPH, title="Banking Network — Attack Surface (10 nodes)")

## 6. Uninformed Search — BFS and DFS

**BFS** explores nodes level by level (FIFO queue). Optimal for unweighted graphs.  
**DFS** explores deeply first (LIFO stack). Not guaranteed optimal.

In [ ]:
def bfs(graph, start, goal):
    """Breadth-First Search."""
    t0 = time.perf_counter()
    queue = deque([start])
    visited = {start}
    came_from = {}
    visited_order = []

    while queue:
        current = queue.popleft()
        visited_order.append(current)
        if current == goal:
            path = graph.reconstruct_path(came_from, goal)
            elapsed = (time.perf_counter() - t0) * 1000
            return SearchResult("BFS", path, graph.path_cost(path),
                                len(visited_order), elapsed, True, "", visited_order)
        for nbr, _ in graph.neighbors(current):
            if nbr not in visited:
                visited.add(nbr)
                came_from[nbr] = current
                queue.append(nbr)

    elapsed = (time.perf_counter() - t0) * 1000
    return SearchResult("BFS", [], float("inf"), len(visited_order), elapsed, False, "No path found")


def dfs(graph, start, goal):
    """Depth-First Search."""
    t0 = time.perf_counter()
    stack = [start]
    visited = set()
    came_from = {}
    visited_order = []

    while stack:
        current = stack.pop()
        if current in visited:
            continue
        visited.add(current)
        visited_order.append(current)
        if current == goal:
            path = graph.reconstruct_path(came_from, goal)
            elapsed = (time.perf_counter() - t0) * 1000
            return SearchResult("DFS", path, graph.path_cost(path),
                                len(visited_order), elapsed, True, "", visited_order)
        for nbr, _ in reversed(graph.neighbors(current)):
            if nbr not in visited:
                if nbr not in came_from:
                    came_from[nbr] = current
                stack.append(nbr)

    elapsed = (time.perf_counter() - t0) * 1000
    return SearchResult("DFS", [], float("inf"), len(visited_order), elapsed, False, "No path found")


r_bfs = bfs(GRAPH, START, GOAL)
r_dfs = dfs(GRAPH, START, GOAL)

print(f"BFS → Path: {' → '.join(r_bfs.path)}")
print(f"      Cost: {r_bfs.cost} | Nodes expanded: {r_bfs.nodes_expanded} | Time: {r_bfs.execution_time_ms:.3f} ms\n")
print(f"DFS → Path: {' → '.join(r_dfs.path)}")
print(f"      Cost: {r_dfs.cost} | Nodes expanded: {r_dfs.nodes_expanded} | Time: {r_dfs.execution_time_ms:.3f} ms")

visualize_network(GRAPH, path=r_bfs.path, title="BFS — Discovered Attack Path")
visualize_network(GRAPH, path=r_dfs.path, title="DFS — Discovered Attack Path")

## 7. Informed Search — UCS and A*

**UCS** expands the lowest-cost frontier node g(n). Optimal for non-negative weights.  
**A*** uses f(n) = g(n) + h(n). Optimal when the heuristic is admissible.

### Heuristic Design for A*
h(n) = estimated number of compromise steps remaining to the Target Database.  
This is pre-computed and stored per node. Because h(n) never overestimates true remaining cost, it is **admissible**, guaranteeing optimality.

In [ ]:
def ucs(graph, start, goal):
    """Uniform-Cost Search — expands minimum g(n)."""
    t0 = time.perf_counter()
    frontier = [(0.0, start)]
    g_score = {start: 0.0}
    came_from = {}
    visited_order = []
    expanded = set()

    while frontier:
        cost, current = heapq.heappop(frontier)
        if current in expanded:
            continue
        expanded.add(current)
        visited_order.append(current)
        if current == goal:
            path = graph.reconstruct_path(came_from, goal)
            elapsed = (time.perf_counter() - t0) * 1000
            return SearchResult("UCS", path, graph.path_cost(path),
                                len(visited_order), elapsed, True, "", visited_order)
        for nbr, weight in graph.neighbors(current):
            new_cost = g_score[current] + weight
            if nbr not in g_score or new_cost < g_score[nbr]:
                g_score[nbr] = new_cost
                came_from[nbr] = current
                heapq.heappush(frontier, (new_cost, nbr))

    elapsed = (time.perf_counter() - t0) * 1000
    return SearchResult("UCS", [], float("inf"), len(visited_order), elapsed, False, "No path found")


def astar(graph, start, goal):
    """A* Search — f(n) = g(n) + h(n)."""
    t0 = time.perf_counter()
    g_score = {start: 0.0}
    frontier = [(g_score[start] + graph.h(start), start)]
    came_from = {}
    visited_order = []
    closed = set()

    while frontier:
        _, current = heapq.heappop(frontier)
        if current in closed:
            continue
        closed.add(current)
        visited_order.append(current)
        if current == goal:
            path = graph.reconstruct_path(came_from, goal)
            elapsed = (time.perf_counter() - t0) * 1000
            return SearchResult("A*", path, graph.path_cost(path),
                                len(visited_order), elapsed, True, "", visited_order)
        for nbr, weight in graph.neighbors(current):
            tentative = g_score[current] + weight
            if tentative < g_score.get(nbr, float("inf")):
                g_score[nbr] = tentative
                came_from[nbr] = current
                heapq.heappush(frontier, (tentative + graph.h(nbr), nbr))

    elapsed = (time.perf_counter() - t0) * 1000
    return SearchResult("A*", [], float("inf"), len(visited_order), elapsed, False, "No path found")


r_ucs   = ucs(GRAPH, START, GOAL)
r_astar = astar(GRAPH, START, GOAL)

print(f"UCS → Path: {' → '.join(r_ucs.path)}")
print(f"      Cost: {r_ucs.cost} | Nodes expanded: {r_ucs.nodes_expanded} | Time: {r_ucs.execution_time_ms:.3f} ms\n")
print(f"A*  → Path: {' → '.join(r_astar.path)}")
print(f"      Cost: {r_astar.cost} | Nodes expanded: {r_astar.nodes_expanded} | Time: {r_astar.execution_time_ms:.3f} ms")

visualize_network(GRAPH, path=r_ucs.path,   title="UCS — Optimal Cost Attack Path")
visualize_network(GRAPH, path=r_astar.path, title="A* — Heuristic-Guided Attack Path")

## 8. Local Search — Hill Climbing

At each step, the algorithm moves to the neighbor with the lowest score(n) = edge_weight + h(n).  
It gets **trapped at local maxima** when no neighbour improves the heuristic.

### Local Maximum Demonstration
A dedicated small network is constructed where the only path to the goal forces the algorithm through a node whose heuristic is worse than the current node — causing Hill Climbing to terminate early without reaching the goal.

In [ ]:
def hill_climbing(graph, start, goal, max_steps=50):
    """
    Hill Climbing — greedy local search using h(n).
    Returns first successful path found (no restarts for pure demo).
    """
    t0 = time.perf_counter()
    current = start
    path = [current]
    visited_order = [current]
    expanded = 1

    for _ in range(max_steps):
        if current == goal:
            elapsed = (time.perf_counter() - t0) * 1000
            return SearchResult("Hill Climbing", path, graph.path_cost(path),
                                expanded, elapsed, True, "", visited_order)

        candidates = [
            (w + graph.h(nbr), nbr)
            for nbr, w in graph.neighbors(current)
            if nbr not in path
        ]
        if not candidates:
            break

        candidates.sort()
        best_score, next_node = candidates[0]

        # Local maximum: no move improves heuristic
        if graph.h(next_node) >= graph.h(current) and next_node != goal:
            elapsed = (time.perf_counter() - t0) * 1000
            return SearchResult("Hill Climbing", path, float("inf"),
                                expanded, elapsed, False,
                                f"Stuck at local maximum: '{current}'", visited_order)

        current = next_node
        path.append(current)
        visited_order.append(current)
        expanded += 1

    elapsed = (time.perf_counter() - t0) * 1000
    success = path[-1] == goal
    return SearchResult("Hill Climbing", path if success else [],
                        graph.path_cost(path) if success else float("inf"),
                        expanded, elapsed, success,
                        "" if success else "Hill Climbing failed to reach goal", visited_order)


# ── Run on main network ──
r_hc = hill_climbing(GRAPH, START, GOAL)
print(f"Hill Climbing on main network:")
print(f"  Success : {r_hc.success}")
print(f"  Path    : {' → '.join(r_hc.path) if r_hc.path else 'N/A'}")
print(f"  Message : {r_hc.message}")

# ── Local Maximum Demonstration ──
print("\n--- Local Maximum Demonstration ---")
trap = NetworkGraph()
trap.add_node("Start",  x=0.0, y=0.5, h=5, ntype="attacker")
trap.add_node("NodeA",  x=0.3, y=0.5, h=3, ntype="host")
trap.add_node("NodeB",  x=0.6, y=0.5, h=6, ntype="host")   # h worsens — trap!
trap.add_node("Goal",   x=1.0, y=0.5, h=0, ntype="target")
trap.add_edge("Start", "NodeA", 2)
trap.add_edge("NodeA", "NodeB", 1)   # only path to Goal goes through NodeB
trap.add_edge("NodeB", "Goal",  1)

r_trap = hill_climbing(trap, "Start", "Goal")
print(f"  Trap network path : {' → '.join(r_trap.path)}")
print(f"  Success           : {r_trap.success}")
print(f"  Message           : {r_trap.message}")
print("  (Hill Climbing stops at NodeA because NodeB has a higher h value)")

visualize_network(GRAPH, path=r_hc.path if r_hc.path else [], title="Hill Climbing — Main Network")

## 9. Adversarial Search — Minimax and Alpha-Beta Pruning

A two-player model:
- **Attacker (MAX):** Tries to reach the Target Database (maximises utility = −path_cost).
- **Defender (MIN):** Tries to block or increase path cost (minimises utility).

**Minimax** exhaustively evaluates the game tree.  
**Alpha-Beta Pruning** cuts branches that cannot affect the final decision, reducing node expansions while producing the same result.

In [ ]:
def _build_game_tree(graph, start, goal, depth=4):
    """Recursively build a shallow game tree from the network."""
    def expand(path, d, is_attacker):
        current = path[-1]
        if current == goal or d == 0:
            cost = graph.path_cost(path) if len(path) > 1 else 0
            utility = -cost if current == goal else -1000
            return {"type": "leaf", "utility": utility, "path": list(path)}
        node = {"type": "max" if is_attacker else "min", "children": []}
        for nbr, w in graph.neighbors(current):
            if nbr in path:
                continue
            child = expand(path + [nbr], d - 1, not is_attacker)
            node["children"].append({"move": nbr, "subtree": child})
        if not node["children"]:
            return {"type": "leaf", "utility": -1000, "path": list(path)}
        return node
    return expand([start], depth, True)


def _minimax_search(node, depth, maximizing, stats):
    stats["nodes_expanded"] += 1
    if node["type"] == "leaf" or depth == 0:
        return node.get("utility", -1000), node.get("path", [])
    best_path = []
    if maximizing:
        best_val = float("-inf")
        for child in node.get("children", []):
            val, path = _minimax_search(child["subtree"], depth-1, False, stats)
            if val > best_val:
                best_val, best_path = val, path
        return best_val, best_path
    else:
        best_val = float("inf")
        for child in node.get("children", []):
            val, path = _minimax_search(child["subtree"], depth-1, True, stats)
            if val < best_val:
                best_val, best_path = val, path
        return best_val, best_path


def _alphabeta_search(node, depth, alpha, beta, maximizing, stats):
    stats["nodes_expanded"] += 1
    if node["type"] == "leaf" or depth == 0:
        return node.get("utility", -1000), node.get("path", [])
    best_path = []
    if maximizing:
        value = float("-inf")
        for child in node.get("children", []):
            val, path = _alphabeta_search(child["subtree"], depth-1, alpha, beta, False, stats)
            if val > value:
                value, best_path = val, path
            alpha = max(alpha, value)
            if beta <= alpha:
                stats["pruned"] += 1
                break
        return value, best_path
    else:
        value = float("inf")
        for child in node.get("children", []):
            val, path = _alphabeta_search(child["subtree"], depth-1, alpha, beta, True, stats)
            if val < value:
                value, best_path = val, path
            beta = min(beta, value)
            if beta <= alpha:
                stats["pruned"] += 1
                break
        return value, best_path


def minimax(graph, start, goal, depth=4):
    t0 = time.perf_counter()
    tree = _build_game_tree(graph, start, goal, depth)
    stats = {"nodes_expanded": 0}
    utility, path = _minimax_search(tree, depth, True, stats)
    elapsed = (time.perf_counter() - t0) * 1000
    success = len(path) > 0 and path[-1] == goal
    cost = graph.path_cost(path) if success else float("inf")
    return SearchResult("Minimax", path, cost, stats["nodes_expanded"],
                        elapsed, success,
                        "" if success else "Could not reach goal within depth",
                        path, {"utility": utility})


def alpha_beta(graph, start, goal, depth=4):
    t0 = time.perf_counter()
    tree = _build_game_tree(graph, start, goal, depth)
    stats = {"nodes_expanded": 0, "pruned": 0}
    utility, path = _alphabeta_search(tree, depth, float("-inf"), float("inf"), True, stats)
    elapsed = (time.perf_counter() - t0) * 1000
    success = len(path) > 0 and path[-1] == goal
    cost = graph.path_cost(path) if success else float("inf")
    return SearchResult("Alpha-Beta", path, cost, stats["nodes_expanded"],
                        elapsed, success,
                        "" if success else "Could not reach goal within depth",
                        path, {"utility": utility, "pruned": stats["pruned"]})


r_mm = minimax(GRAPH, START, GOAL, depth=4)
r_ab = alpha_beta(GRAPH, START, GOAL, depth=4)

print(f"Minimax    → Path: {' → '.join(r_mm.path) if r_mm.path else 'N/A'}")
print(f"             Nodes expanded: {r_mm.nodes_expanded} | Time: {r_mm.execution_time_ms:.3f} ms")
print(f"Alpha-Beta → Path: {' → '.join(r_ab.path) if r_ab.path else 'N/A'}")
print(f"             Nodes expanded: {r_ab.nodes_expanded} | Pruned: {r_ab.extra.get('pruned',0)} | Time: {r_ab.execution_time_ms:.3f} ms")
print(f"\nAlpha-Beta pruned {r_ab.extra.get('pruned',0)} branches vs Minimax.")

visualize_network(GRAPH, path=r_mm.path, title="Minimax — Attacker vs Defender Path")
visualize_network(GRAPH, path=r_ab.path, title="Alpha-Beta Pruning — Optimised Adversarial Path")

## 10. Comparative Analysis Table

All algorithms run on the **same network** with the **same start and goal nodes**.

In [ ]:
all_results = [r_bfs, r_dfs, r_ucs, r_astar, r_hc, r_mm, r_ab]

table_data = []
for r in all_results:
    cost_str = str(r.cost) if r.cost != float("inf") else "∞ (failed)"
    path_str = " → ".join(r.path) if r.path else "No path"
    table_data.append([
        r.algorithm,
        path_str,
        cost_str,
        r.nodes_expanded,
        f"{r.execution_time_ms:.3f}",
        "✓" if r.success else "✗"
    ])

headers = ["Algorithm", "Discovered Path", "Total Cost", "Nodes Expanded", "Time (ms)", "Success"]
print(tabulate(table_data, headers=headers, tablefmt="grid"))

## 11. Performance Comparison Charts

In [ ]:
labels   = [r.algorithm for r in all_results]
costs    = [r.cost if r.cost != float("inf") else 0 for r in all_results]
expanded = [r.nodes_expanded for r in all_results]
times    = [r.execution_time_ms for r in all_results]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Algorithm Performance Comparison", fontsize=13, fontweight="bold")

colors = ["#E24B4A","#7F77DD","#1D9E75","#EF9F27","#378ADD","#D85A30","#0F6E56"]

for ax, data, ylabel, title in zip(
    axes,
    [costs, expanded, times],
    ["Path Cost", "Nodes Expanded", "Execution Time (ms)"],
    ["Path Cost", "Nodes Expanded", "Execution Time (ms)"]
):
    bars = ax.bar(labels, data, color=colors, edgecolor="white", linewidth=0.5)
    ax.set_title(title, fontsize=11)
    ax.set_ylabel(ylabel, fontsize=9)
    ax.set_xticklabels(labels, rotation=30, ha="right", fontsize=8)
    for bar, val in zip(bars, data):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                f"{val:.1f}", ha="center", va="bottom", fontsize=8)

plt.tight_layout()
plt.show()

## 12. Export Network to JSON (Dynamic Loading Support)

In [ ]:
network_data = {
    "nodes": [
        {
            "id": n,
            "x": GRAPH.positions[n][0],
            "y": GRAPH.positions[n][1],
            "h": GRAPH.heuristics[n],
            "type": GRAPH.node_types[n]
        }
        for n in GRAPH.nodes
    ],
    "edges": [
        {"from": u, "to": v, "weight": w}
        for u in GRAPH.nodes
        for v, w in GRAPH.neighbors(u)
    ],
    "start": START,
    "goal": GOAL
}

with open("network_topology.json", "w") as f:
    json.dump(network_data, f, indent=2)

print("Network exported to network_topology.json")
print(json.dumps(network_data, indent=2)[:600], "...")

## 13. Summary and Findings

| Algorithm | Type | Optimal? | Notes |
|-----------|------|----------|-------|
| BFS | Uninformed | Only for unweighted | Explores all nodes level by level |
| DFS | Uninformed | No | May find suboptimal paths; low memory |
| UCS | Informed | Yes | Guarantees minimum-cost path |
| A* | Informed | Yes (admissible h) | Fastest optimal; guided by heuristic |
| Hill Climbing | Local | No | Greedy; gets trapped at local maxima |
| Minimax | Adversarial | Yes (within depth) | Models attacker vs defender |
| Alpha-Beta | Adversarial | Yes (within depth) | Same result as Minimax, fewer expansions |

**Key takeaways:**
- A* expands the fewest nodes among optimal algorithms thanks to the admissible heuristic.
- Hill Climbing is fast but can fail — demonstrated with the local-maximum trap network.
- Alpha-Beta Pruning significantly reduces Minimax node expansions while producing the same path.
- UCS and A* both find the optimal path; A* does so more efficiently.

---
## 14. Interactive UI — Graphical User Interface (ipywidgets)

This section provides a **full interactive GUI** that runs directly inside Google Colab.  
Use the buttons and dropdown below to:
- Select any algorithm
- Click **▶ Run Algorithm** to execute it
- See the discovered path highlighted on the network graph
- View live metrics (cost, nodes expanded, execution time)
- Build the full comparison table by running all algorithms

> **Note:** Run all cells above first so the graph and algorithms are loaded.

In [ ]:
# Install ipywidgets (pre-installed in Colab, this ensures it is enabled)
!pip install ipywidgets --quiet
print("ipywidgets ready.")

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx

# ─────────────────────────────────────────────────────────────────
#  COLOUR SCHEME
# ─────────────────────────────────────────────────────────────────
TYPE_COLORS = {
    'attacker':    '#E24B4A',
    'target':      '#1D9E75',
    'firewall':    '#EF9F27',
    'workstation': '#378ADD',
    'atm':         '#7F77DD',
    'auth':        '#378ADD',
    'monitor':     '#EF9F27',
    'transaction': '#378ADD',
    'database':    '#7F77DD',
    'backup':      '#888780',
    'host':        '#888780',
}

# ─────────────────────────────────────────────────────────────────
#  GRAPH DRAWING  (start/goal highlighted dynamically)
# ─────────────────────────────────────────────────────────────────
def draw_graph_ui(graph, path=None, title='Network Topology',
                  ax=None, start_node=None, goal_node=None):
    G = nx.DiGraph()
    for node in graph.nodes:
        G.add_node(node)
    for u in graph.nodes:
        for v, w in graph.neighbors(u):
            G.add_edge(u, v, weight=w)

    pos = {n: graph.positions[n] for n in graph.nodes}

    # Colour nodes: start=red, goal=green, path=lighter tint, others=type colour
    node_colors = []
    for n in G.nodes():
        if n == start_node:
            node_colors.append('#E24B4A')
        elif n == goal_node:
            node_colors.append('#1D9E75')
        elif path and n in path:
            node_colors.append('#FFA500')
        else:
            node_colors.append(TYPE_COLORS.get(graph.node_types.get(n, 'host'), '#888780'))

    if ax is None:
        ax = plt.gca()

    ax.set_facecolor('#f8f9fc')
    ax.set_title(title, fontsize=12, fontweight='bold', pad=10, color='#1a1a2e')

    # All edges (grey)
    nx.draw_networkx_edges(G, pos, ax=ax, edge_color='#cccccc', arrows=True,
                           arrowsize=14, width=1.2, connectionstyle='arc3,rad=0.1')

    # Highlighted path edges (red)
    if path and len(path) > 1:
        path_edges = [(path[i], path[i+1]) for i in range(len(path)-1)]
        nx.draw_networkx_edges(G, pos, edgelist=path_edges, ax=ax,
                               edge_color='#E24B4A', arrows=True,
                               arrowsize=20, width=3, connectionstyle='arc3,rad=0.1')

    nx.draw_networkx_nodes(G, pos, ax=ax, node_color=node_colors,
                           node_size=1400, alpha=0.95)
    nx.draw_networkx_labels(G, pos, ax=ax, font_size=6.5,
                            font_weight='bold', font_color='white')

    edge_labels = {(u, v): d['weight'] for u, v, d in G.edges(data=True)}
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels,
                                 ax=ax, font_size=7, label_pos=0.35)

    legend_elements = [
        mpatches.Patch(color='#E24B4A', label='Start Node'),
        mpatches.Patch(color='#1D9E75', label='Goal Node'),
        mpatches.Patch(color='#FFA500', label='Path Node'),
        mpatches.Patch(color='#EF9F27', label='Firewall / Monitor'),
        mpatches.Patch(color='#7F77DD', label='Database / ATM'),
        mpatches.Patch(color='#378ADD', label='Server / Auth'),
        mpatches.Patch(color='#888780', label='Backup / Host'),
    ]
    ax.legend(handles=legend_elements, loc='lower left', fontsize=7.5,
              framealpha=0.85, edgecolor='#cccccc')
    ax.axis('off')


# ─────────────────────────────────────────────────────────────────
#  STATE
# ─────────────────────────────────────────────────────────────────
results_store = {}
ALL_NODES = list(GRAPH.nodes)   # populated from the network built in Section 4

ALGO_MAP_TEMPLATE = {
    'BFS  - Breadth-First Search':  bfs,
    'DFS  - Depth-First Search':    dfs,
    'UCS  - Uniform Cost Search':   ucs,
    'A*   - A-Star Search':         astar,
    'HC   - Hill Climbing':         hill_climbing,
    'MM   - Minimax (depth=4)':     lambda g, s, gl: minimax(g, s, gl, depth=4),
    'AB   - Alpha-Beta (depth=4)':  lambda g, s, gl: alpha_beta(g, s, gl, depth=4),
}

def run_algo(algo_key, start, goal):
    fn = ALGO_MAP_TEMPLATE[algo_key]
    return fn(GRAPH, start, goal)


# ─────────────────────────────────────────────────────────────────
#  TITLE BANNER
# ─────────────────────────────────────────────────────────────────
title_html = widgets.HTML(value="""
<div style='background:#1a1a2e;padding:14px 20px;border-radius:8px;margin-bottom:8px'>
  <h2 style='color:#e94560;margin:0;font-family:monospace;font-size:16px;letter-spacing:1px'>
    AI-Based Network Pathfinding &amp; Attack Simulation
  </h2>
  <p style='color:#a8b2d8;margin:4px 0 0;font-size:12px;font-family:monospace'>
    CSC 262 &nbsp;|&nbsp; COMSATS University Islamabad &nbsp;|&nbsp; Interactive GUI
  </p>
</div>""")

# ─────────────────────────────────────────────────────────────────
#  ROW 1: START NODE selector
# ─────────────────────────────────────────────────────────────────
start_label = widgets.HTML(
    value="<b style='font-family:monospace;font-size:12px;color:#1a1a2e'>Start Node (Attacker):</b>")
start_dropdown = widgets.Dropdown(
    options=ALL_NODES,
    value='Attacker Entry',
    layout=widgets.Layout(width='220px'),
    style={'description_width': '0px'}
)

goal_label = widgets.HTML(
    value="<b style='font-family:monospace;font-size:12px;color:#1a1a2e'>Goal Node (Target):</b>")
goal_dropdown = widgets.Dropdown(
    options=ALL_NODES,
    value='Target Database',
    layout=widgets.Layout(width='220px'),
    style={'description_width': '0px'}
)

node_row = widgets.HBox(
    [start_label, start_dropdown,
     widgets.HTML("&nbsp;&nbsp;&nbsp;"),
     goal_label,  goal_dropdown],
    layout=widgets.Layout(
        align_items='center', flex_wrap='wrap', gap='6px',
        padding='8px 10px', background='#f0f4ff',
        border='1px solid #c5cde8', border_radius='6px', margin_bottom='6px'
    )
)

# ─────────────────────────────────────────────────────────────────
#  ROW 2: ALGORITHM selector + buttons
# ─────────────────────────────────────────────────────────────────
algo_label_w = widgets.HTML(
    value="<b style='font-family:monospace;font-size:12px;color:#1a1a2e'>Algorithm:</b>")
algo_dropdown = widgets.Dropdown(
    options=list(ALGO_MAP_TEMPLATE.keys()),
    value=list(ALGO_MAP_TEMPLATE.keys())[0],
    layout=widgets.Layout(width='280px'),
    style={'description_width': '0px'}
)

btn_run     = widgets.Button(description='Run Algorithm', button_style='success',
                              layout=widgets.Layout(width='150px', height='36px'))
btn_run_all = widgets.Button(description='Run All',       button_style='warning',
                              layout=widgets.Layout(width='110px', height='36px'))
btn_clear   = widgets.Button(description='Clear',         button_style='danger',
                              layout=widgets.Layout(width='90px',  height='36px'))
status_out  = widgets.HTML(
    value="<span style='font-family:monospace;font-size:12px;color:#888'>Status: READY</span>")

controls_row = widgets.HBox(
    [algo_label_w, algo_dropdown, btn_run, btn_run_all, btn_clear, status_out],
    layout=widgets.Layout(align_items='center', flex_wrap='wrap', gap='8px', padding='6px 0')
)

# ─────────────────────────────────────────────────────────────────
#  METRICS + PATH OUTPUT
# ─────────────────────────────────────────────────────────────────
metrics_out = widgets.HTML(value="""
<div style='background:#f0f4ff;border:1px solid #c5cde8;border-radius:6px;
            padding:12px 16px;font-family:monospace;min-width:260px'>
  <b style='color:#555'>Metrics</b><br><br>
  <span style='color:#999'>Run an algorithm to see results.</span>
</div>""")

path_out = widgets.HTML(value="""
<div style='background:#fff8e1;border:1px solid #ffe082;border-radius:6px;
            padding:10px 14px;font-family:monospace;font-size:12px'>
  <b>Discovered Path:</b> <span style='color:#999'>none yet</span>
</div>""")

graph_out = widgets.Output(layout=widgets.Layout(width='100%'))
table_out = widgets.Output()


# ─────────────────────────────────────────────────────────────────
#  HELPER FUNCTIONS
# ─────────────────────────────────────────────────────────────────
def make_metrics_html(r, start, goal):
    ok_color = '#1D9E75' if r.success else '#E24B4A'
    ok_label = 'SUCCESS' if r.success else 'FAILED'
    cost_str = str(r.cost) if r.cost != float('inf') else 'No path'
    return f"""
<div style='background:#f0f4ff;border:1px solid #c5cde8;border-radius:6px;
            padding:12px 16px;font-family:monospace;min-width:260px'>
  <b style='font-size:13px;color:#1a1a2e'>{r.algorithm}</b><br>
  <span style='font-size:11px;color:#888'>{start} &rarr; {goal}</span><br><br>
  <table style='border-collapse:collapse;width:100%;font-size:12px'>
    <tr><td style='color:#666;padding:3px 0'>Path Cost</td>
        <td style='font-weight:bold;text-align:right;color:#1a1a2e'>{cost_str}</td></tr>
    <tr><td style='color:#666;padding:3px 0'>Nodes Expanded</td>
        <td style='font-weight:bold;text-align:right;color:#1a1a2e'>{r.nodes_expanded}</td></tr>
    <tr><td style='color:#666;padding:3px 0'>Time (ms)</td>
        <td style='font-weight:bold;text-align:right;color:#1a1a2e'>{r.execution_time_ms:.4f}</td></tr>
    <tr><td style='color:#666;padding:3px 0'>Status</td>
        <td style='font-weight:bold;text-align:right;color:{ok_color}'>{ok_label}</td></tr>
  </table>
</div>"""


def make_path_html(r, start, goal):
    if not r.path:
        msg = r.message or 'No path found between selected nodes'
        return f"""
<div style='background:#fff0f0;border:1px solid #ffb3b3;border-radius:6px;
            padding:10px 14px;font-family:monospace;font-size:12px'>
  <b>Discovered Path:</b><br>
  <span style='color:#E24B4A'>{msg}</span>
</div>"""
    parts = []
    for node in r.path:
        if node == start:
            color = '#E24B4A'
        elif node == goal:
            color = '#1D9E75'
        else:
            color = '#378ADD'
        parts.append(
            f"<span style='background:{color};color:white;padding:2px 8px;"
            f"border-radius:4px;font-size:11px;white-space:nowrap'>{node}</span>"
        )
    nodes_html = " <span style='color:#aaa'>&rarr;</span> ".join(parts)
    return f"""
<div style='background:#fff8e1;border:1px solid #ffe082;border-radius:6px;
            padding:10px 14px;font-family:monospace;font-size:12px;line-height:2.2'>
  <b>Discovered Path:</b><br>{nodes_html}
</div>"""


def render_table(active_algo=None):
    if not results_store:
        return
    rows = ''
    for key, r in results_store.items():
        s, g    = r.extra.get('_start', '?'), r.extra.get('_goal', '?')
        cost_s  = str(r.cost) if r.cost != float('inf') else 'None'
        ok_col  = '#1D9E75' if r.success else '#E24B4A'
        ok_sym  = 'YES' if r.success else 'NO'
        path_s  = ' -> '.join(r.path) if r.path else '-'
        if len(path_s) > 55:
            path_s = path_s[:52] + '...'
        is_active = (key == active_algo)
        row_bg = "background:#fffff0;" if is_active else ""
        rows += f"""
<tr style='{row_bg}'>
  <td style='padding:6px 8px;border-bottom:1px solid #eee;font-weight:bold;white-space:nowrap'>{r.algorithm}</td>
  <td style='padding:6px 8px;border-bottom:1px solid #eee;font-size:11px;color:#555;white-space:nowrap'>{s} &rarr; {g}</td>
  <td style='padding:6px 8px;border-bottom:1px solid #eee;text-align:center'>{cost_s}</td>
  <td style='padding:6px 8px;border-bottom:1px solid #eee;text-align:center'>{r.nodes_expanded}</td>
  <td style='padding:6px 8px;border-bottom:1px solid #eee;text-align:center'>{r.execution_time_ms:.4f}</td>
  <td style='padding:6px 8px;border-bottom:1px solid #eee;text-align:center;color:{ok_col};font-weight:bold'>{ok_sym}</td>
  <td style='padding:6px 8px;border-bottom:1px solid #eee;font-size:10px;color:#555'>{path_s}</td>
</tr>"""

    html = f"""
<div style='font-family:monospace'>
  <b style='font-size:13px'>Comparative Analysis Table</b><br><br>
  <div style='overflow-x:auto'>
  <table style='border-collapse:collapse;width:100%;font-size:12px;border:1px solid #ddd'>
    <thead>
      <tr style='background:#1a1a2e;color:white'>
        <th style='padding:8px;text-align:left'>Algorithm</th>
        <th style='padding:8px;text-align:left'>Start &rarr; Goal</th>
        <th style='padding:8px'>Cost</th>
        <th style='padding:8px'>Nodes</th>
        <th style='padding:8px'>Time(ms)</th>
        <th style='padding:8px'>Found?</th>
        <th style='padding:8px;text-align:left'>Path</th>
      </tr>
    </thead>
    <tbody>{rows}</tbody>
  </table>
  </div>
</div>"""
    with table_out:
        clear_output(wait=True)
        display(widgets.HTML(value=html))


def redraw_graph(path, title, start, goal):
    with graph_out:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(13, 6))
        draw_graph_ui(GRAPH, path=path, title=title, ax=ax,
                      start_node=start, goal_node=goal)
        plt.tight_layout()
        plt.show()


# ─────────────────────────────────────────────────────────────────
#  VALIDATION HELPER
# ─────────────────────────────────────────────────────────────────
def validate_nodes(start, goal):
    """Returns (ok, error_message)"""
    if start == goal:
        return False, "Start and Goal nodes must be different!"
    return True, ""


# ─────────────────────────────────────────────────────────────────
#  BUTTON CALLBACKS
# ─────────────────────────────────────────────────────────────────
def on_run(b):
    start = start_dropdown.value
    goal  = goal_dropdown.value
    ok, err = validate_nodes(start, goal)
    if not ok:
        status_out.value = f"<span style='font-family:monospace;font-size:12px;color:#E24B4A'>ERROR: {err}</span>"
        return

    algo_key = algo_dropdown.value
    status_out.value = "<span style='font-family:monospace;font-size:12px;color:#EF9F27'>Status: RUNNING...</span>"

    result = run_algo(algo_key, start, goal)
    # Attach start/goal to result for table display
    result.extra['_start'] = start
    result.extra['_goal']  = goal

    # Store with a unique key = algo + start + goal
    store_key = f"{result.algorithm}|{start}|{goal}"
    results_store[store_key] = result

    metrics_out.value = make_metrics_html(result, start, goal)
    path_out.value    = make_path_html(result, start, goal)
    redraw_graph(result.path,
                 f"{result.algorithm}  |  {start}  ->  {goal}",
                 start, goal)
    render_table(active_algo=store_key)

    ok_text = 'PATH FOUND' if result.success else 'NO PATH'
    ok_col  = '#1D9E75'    if result.success else '#E24B4A'
    status_out.value = f"<span style='font-family:monospace;font-size:12px;color:{ok_col}'>Status: {ok_text}</span>"


def on_run_all(b):
    start = start_dropdown.value
    goal  = goal_dropdown.value
    ok, err = validate_nodes(start, goal)
    if not ok:
        status_out.value = f"<span style='font-family:monospace;font-size:12px;color:#E24B4A'>ERROR: {err}</span>"
        return

    status_out.value = "<span style='font-family:monospace;font-size:12px;color:#EF9F27'>Status: RUNNING ALL...</span>"
    last = None
    last_key = None
    for key in ALGO_MAP_TEMPLATE:
        r = run_algo(key, start, goal)
        r.extra['_start'] = start
        r.extra['_goal']  = goal
        store_key = f"{r.algorithm}|{start}|{goal}"
        results_store[store_key] = r
        last     = r
        last_key = store_key

    metrics_out.value = make_metrics_html(last, start, goal)
    path_out.value    = make_path_html(last, start, goal)
    redraw_graph(last.path,
                 f"All Algorithms Run  |  {start}  ->  {goal}  (last result shown)",
                 start, goal)
    render_table(active_algo=last_key)
    status_out.value = "<span style='font-family:monospace;font-size:12px;color:#1D9E75'>Status: ALL DONE</span>"


def on_clear(b):
    results_store.clear()
    metrics_out.value = """
<div style='background:#f0f4ff;border:1px solid #c5cde8;border-radius:6px;
            padding:12px 16px;font-family:monospace;min-width:260px'>
  <b style='color:#555'>Metrics</b><br><br>
  <span style='color:#999'>Run an algorithm to see results.</span>
</div>"""
    path_out.value = """
<div style='background:#fff8e1;border:1px solid #ffe082;border-radius:6px;
            padding:10px 14px;font-family:monospace;font-size:12px'>
  <b>Discovered Path:</b> <span style='color:#999'>none yet</span>
</div>"""
    s = start_dropdown.value
    g = goal_dropdown.value
    redraw_graph([], f'Network Topology  |  Start: {s}  |  Goal: {g}', s, g)
    with table_out:
        clear_output()
    status_out.value = "<span style='font-family:monospace;font-size:12px;color:#888'>Status: CLEARED</span>"


def on_node_change(change):
    """Redraw graph whenever start/goal selection changes."""
    s = start_dropdown.value
    g = goal_dropdown.value
    redraw_graph([], f'Network Topology  |  Start: {s}  |  Goal: {g}', s, g)
    results_store.clear()
    with table_out:
        clear_output()
    path_out.value = """
<div style='background:#fff8e1;border:1px solid #ffe082;border-radius:6px;
            padding:10px 14px;font-family:monospace;font-size:12px'>
  <b>Discovered Path:</b> <span style='color:#999'>none yet — nodes changed</span>
</div>"""
    metrics_out.value = """
<div style='background:#f0f4ff;border:1px solid #c5cde8;border-radius:6px;
            padding:12px 16px;font-family:monospace;min-width:260px'>
  <b style='color:#555'>Metrics</b><br><br>
  <span style='color:#999'>Run an algorithm to see results.</span>
</div>"""
    status_out.value = "<span style='font-family:monospace;font-size:12px;color:#888'>Status: NODES CHANGED</span>"


btn_run.on_click(on_run)
btn_run_all.on_click(on_run_all)
btn_clear.on_click(on_clear)
start_dropdown.observe(on_node_change, names='value')
goal_dropdown.observe(on_node_change,  names='value')


# ─────────────────────────────────────────────────────────────────
#  LAYOUT ASSEMBLY
# ─────────────────────────────────────────────────────────────────
right_panel = widgets.VBox(
    [metrics_out, widgets.HTML('<br>'), path_out],
    layout=widgets.Layout(min_width='280px', padding='0 0 0 12px')
)

main_row = widgets.HBox(
    [graph_out, right_panel],
    layout=widgets.Layout(align_items='flex-start', width='100%')
)

table_section = widgets.VBox(
    [widgets.HTML("<hr style='border:none;border-top:1px solid #ddd;margin:10px 0'>"),
     table_out],
    layout=widgets.Layout(width='100%')
)

ui = widgets.VBox(
    [title_html, node_row, controls_row, main_row, table_section],
    layout=widgets.Layout(padding='12px', width='100%', border='1px solid #e0e0e0')
)

# Initial graph draw
s0 = start_dropdown.value
g0 = goal_dropdown.value
redraw_graph([], f'Network Topology  |  Start: {s0}  |  Goal: {g0}', s0, g0)

display(ui)
print("UI loaded!  Change Start / Goal dropdowns, select an algorithm, then click Run.")
